# Notebook 01 — Occurrence Ratings from Production Data

**Project:** NEXUS-FMEA (Data-Driven PFMEA + Linked Control Plan)  
**Data source:** CiP-DMD dataset via Project 1 (Sentinel-8D)  
**Purpose:** Load the per-part quality data from P1 and compute per-operation /
per-characteristic defect rates to derive AIAG-VDA Occurrence ratings.

---

**Day 1 scope:** Load the data, verify its shape, and confirm it's ready for
defect-rate computation on Day 2.

## 1 — Load Project 1 Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/defect_rates.csv")
print(f"Shape: {df.shape[0]} parts × {df.shape[1]} columns")
df.head()

## 2 — Column inventory & data types

The key columns for PFMEA Occurrence are the `_qcpass` flags — one per
quality characteristic per manufacturing operation. `True` = passed QC,
`False` = defect detected.

In [ ]:
# Data types overview
print("=== Data types ===")
print(df.dtypes.to_string())
print(f"\n=== Null counts ===")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "No nulls")

## 3 — Sanity check: per-characteristic defect counts

Quick look at how many parts failed each QC characteristic.
These counts will become Occurrence numerators on Day 2.

In [ ]:
# Per-characteristic defect counts
qcpass_cols = [c for c in df.columns if c.endswith('_qcpass')]

print(f"{'Characteristic':<35} {'Total':>6} {'Pass':>6} {'Fail':>6} {'Fail %':>8}")
print("-" * 67)
for col in qcpass_cols:
    total = df[col].notna().sum()
    # Convert string 'True'/'False' to boolean if needed
    pass_count = (df[col].astype(str) == 'True').sum()
    fail_count = (df[col].astype(str) == 'False').sum()
    fail_pct = fail_count / total * 100 if total > 0 else 0
    char_name = col.replace('_qcpass', '')
    print(f"{char_name:<35} {total:>6} {pass_count:>6} {fail_count:>6} {fail_pct:>7.2f}%")

print(f"\n{'Overall part fail rate':<35} {len(df):>6} {(df['fail']==0).sum():>6} {(df['fail']==1).sum():>6} {df['fail'].mean()*100:>7.2f}%")

## 4 — Process flow: operations in the manufacturing sequence

The CiP-DMD data covers a hydraulic cylinder manufacturing process with
4 operations. This mapping will feed directly into the PFMEA process-flow
column (Day 3):

| Step | Operation | Characteristics measured |
|------|-----------|-------------------------|
| 10   | Sawing    | weight |
| 20   | Milling   | surface roughness, parallelism, groove depth, groove diameter |
| 30   | CNC Lathe | coaxiality, diameter, length |
| 40   | Assembly  | pressure |

In [ ]:
# Map characteristics to manufacturing operations
# This mapping is central to the PFMEA — it ties each failure mode to its process step
OPERATION_MAP = {
    'Sawing':   ['saw_weight'],
    'Milling':  ['mill_surface_roughness', 'mill_parallelism', 'mill_groove_depth', 'mill_groove_diameter'],
    'CNC Lathe': ['lathe_coaxiality', 'lathe_diameter', 'lathe_length'],
    'Assembly': ['assembly_pressure'],
}

# Verify all qcpass columns are accounted for
mapped = [f"{c}_qcpass" for chars in OPERATION_MAP.values() for c in chars]
unmapped = set(qcpass_cols) - set(mapped)
print(f"Mapped {len(mapped)} of {len(qcpass_cols)} QC characteristics")
if unmapped:
    print(f"⚠ Unmapped: {unmapped}")
else:
    print("✅ All characteristics mapped to operations")

## Next: Day 2

Data is loaded and verified. On Day 2 we will:
1. Compute per-operation and per-characteristic defect rates
2. Apply Wilson confidence intervals (`statsmodels`)
3. Map rates to AIAG-VDA Occurrence ratings (1–10 scale)